# Model 3 — 03 Train Models

## Purpose
Train two supervised models:
1. BENIGN vs ATTACK
2. ATTACK family classification

The preprocessing is embedded in each sklearn Pipeline so production inference cannot accidentally use different imputation or scaling logic.

In [1]:

from pathlib import Path
import sys, json, os, warnings
warnings.filterwarnings("ignore")

PROJECT_ROOT = Path.cwd()
while PROJECT_ROOT.name != "EVIDENT" and PROJECT_ROOT.parent != PROJECT_ROOT:
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_DIR = PROJECT_ROOT / "data" / "raw" / "behaviour"
MODEL_DIR = PROJECT_ROOT / "models" / "behavior"
ENV_DIR = MODEL_DIR / "environments"
META_DIR = MODEL_DIR / "metadata"
PROD_DIR = MODEL_DIR / "production"

for d in (ENV_DIR, META_DIR, PROD_DIR):
    d.mkdir(parents=True, exist_ok=True)

if not DATA_DIR.exists():
    raise FileNotFoundError(f"Dataset folder not found: {DATA_DIR}")

import pandas as pd
import numpy as np

def clean_col(c):
    return " ".join(str(c).strip().replace("\ufeff","").split())

def norm_label(x):
    return "UNKNOWN" if pd.isna(x) else str(x).strip().upper().replace(" ","_")

def family(x):
    x = norm_label(x)
    if x == "BENIGN": return "BENIGN"
    if "PATATOR" in x or "BRUTE" in x: return "BRUTE_FORCE"
    if "PORTSCAN" in x or "PORT_SCAN" in x: return "PORT_SCAN"
    if "DDOS" in x or x.startswith("DOS"): return "DOS_DDOS"
    if "WEB_ATTACK" in x or "SQL_INJECTION" in x or "XSS" in x: return "WEB_ATTACK"
    if "BOT" in x: return "BOTNET"
    if "INFILTRATION" in x: return "INFILTRATION"
    if "HEARTBLEED" in x: return "HEARTBLEED"
    return "OTHER_ATTACK"

files = sorted(DATA_DIR.glob("*.csv"))
print("EVIDENT root:", PROJECT_ROOT)
print("CIC-IDS2017 files:", len(files))
for p in files:
    print(" -", p.name)
assert files, "No CIC-IDS2017 CSV files found."


EVIDENT root: c:\Users\bimal\OneDrive\Desktop\EVIDENT
CIC-IDS2017 files: 8
 - Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv
 - Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv
 - Friday-WorkingHours-Morning.pcap_ISCX.csv
 - Monday-WorkingHours.pcap_ISCX.csv
 - Thursday-WorkingHours-Afternoon-Infilteration.pcap_ISCX.csv
 - Thursday-WorkingHours-Morning-WebAttacks.pcap_ISCX.csv
 - Tuesday-WorkingHours.pcap_ISCX.csv
 - Wednesday-workingHours.pcap_ISCX.csv


In [2]:

import pandas as pd, json
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import HistGradientBoostingClassifier

cache = PROJECT_ROOT/"data"/"processed"/"behavior"/"cicids2017_model3_training.parquet"
schema = json.loads((META_DIR/"feature_schema.json").read_text())
df = pd.read_parquet(cache)
features = schema["feature_columns"]

X = df[features]
yb = df["is_attack"]
yf = df["attack_family"]

X_train, X_test, yb_train, yb_test, yf_train, yf_test = train_test_split(
    X, yb, yf, test_size=0.20, random_state=42, stratify=yb
)

def make_model(max_iter=250):
    return Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
        ("classifier", HistGradientBoostingClassifier(
            max_iter=max_iter,
            learning_rate=0.08,
            max_leaf_nodes=31,
            l2_regularization=1.0,
            random_state=42
        ))
    ])

binary_model = make_model(250)
family_model = make_model(300)

print("Training binary detector...")
binary_model.fit(X_train, yb_train)

attack_train = yf_train != "BENIGN"
print("Training attack-family detector...")
family_model.fit(X_train[attack_train], yf_train[attack_train])

print("Training complete.")


Training binary detector...
Training attack-family detector...
Training complete.


In [3]:

from sklearn.metrics import accuracy_score, precision_recall_fscore_support

pred = binary_model.predict(X_test)
p,r,f,_ = precision_recall_fscore_support(yb_test,pred,average="binary",zero_division=0)
print({
    "accuracy": round(accuracy_score(yb_test,pred),4),
    "precision": round(p,4),
    "recall": round(r,4),
    "f1": round(f,4)
})


{'accuracy': 0.9984, 'precision': 0.9987, 'recall': 0.9993, 'f1': 0.999}
